# MatterGen Tutorial for Computational Chemistry Students

This notebook is a **conceptual + practical deep dive** into MatterGen, aimed at students who already know core computational chemistry ideas (DFT, crystal structures, convex hulls, etc.) but are new to modern diffusion-based generative models for materials.

By the end, you should be able to:

1. Explain the **theory** behind MatterGen's diffusion formulation for periodic crystals.
2. Understand the **architecture** (GemNet-style message passing + multi-head denoising).
3. Reproduce the high-level **training workflow** (data curation, preprocessing, training, fine-tuning).
4. Run **unconditional and conditioned generation** with existing checkpoints.
5. **Visualize and inspect** generated crystals from `.extxyz` / `.cif` outputs.

> The tutorial intentionally emphasizes *methodology* and *scientific reasoning* so you can connect model behavior to physical intuition and downstream validation workflows.

## 0) Prerequisites and expected background

You should be comfortable with:

- Basic crystallography: lattice vectors, fractional coordinates, periodic boundary conditions.
- Structure-property modeling concepts: stability, energy above hull, novelty/uniqueness metrics.
- Python basics and command line usage.

Optional but helpful:

- Familiarity with diffusion probabilistic models.
- Experience with graph neural networks and geometric message passing.

This notebook assumes the `mattergen` repository is already cloned and installed in editable mode:

```bash
uv venv .venv --python 3.10
source .venv/bin/activate
uv pip install -e .
```

## 1) Why a diffusion model for inorganic crystal generation?

In computational chemistry and materials design, direct inverse design is difficult because:

- Crystals must satisfy **periodic geometric constraints**.
- Composition, lattice, and atomic positions are **strongly coupled**.
- Useful candidates occupy a tiny manifold inside combinatorial structure space.

MatterGen treats generation as **iterative denoising**: start from noise and repeatedly refine a crystal state toward realistic structures.

### 1.1 Representation in MatterGen

At generation time, MatterGen jointly models:

- Atomic fractional coordinates
- Atomic species (for full generation mode)
- Lattice parameters / cell representation

This is important because sampling only coordinates with fixed species (or vice versa) generally underfits real chemistry. A joint model can better capture correlations such as: preferred coordination motifs, element-size effects on lattice constants, and composition-dependent geometric distortions.

### 1.2 Diffusion intuition

Diffusion training learns to reverse a corruption process. For crystals, corruption is applied to structure variables over a noise schedule, and the network learns denoising fields. During sampling, one traverses from high noise to low noise using a predictor-corrector style algorithm (as implemented in MatterGen's sampler).

From a chemistry lens, each denoising step can be interpreted as gradually injecting physically plausible local ordering and long-range periodic consistency.

## 2) Model architecture and training objective

MatterGen's core architecture is based on **GemNet-style geometric message passing** and is trained as a diffusion denoiser over crystal variables.

### 2.1 Architectural components (high-level)

1. **Geometric graph construction** over atoms in periodic cells.
2. **Message-passing network** encoding directional and distance-aware interactions.
3. **Denoising heads** that predict updates for noisy crystal variables.
4. Optional **property-conditioning pathway** (for fine-tuned checkpoints).

### 2.2 Why geometric message passing matters

In crystals, local chemistry depends on bond geometry and periodic images. A simple composition encoder cannot recover geometry-dependent stability trends. GemNet-like directional features improve expressivity for angular chemistry and local motif reconstruction, which is crucial for producing realistic short-range order during denoising.

### 2.3 Conditional generation mechanism

Conditioned checkpoints are fine-tuned so the model can accept target properties (e.g., magnetic density, band gap proxy, space group, chemical system). Sampling uses a **diffusion guidance factor** (`γ`) akin to classifier-free guidance:

- `γ = 0`: effectively unconditional behavior
- moderate `γ`: stronger property adherence
- large `γ`: can overconstrain and reduce diversity / realism

In practice, students should treat `γ` as a controllable exploration-exploitation knob.

## 3) Data curation and preprocessing pipeline (what is being learned?)

MatterGen training data combines MP and Alexandria sources with strict filtering and preprocessing. Key points:

- Structures are filtered by quality / stability criteria (e.g., energy-above-hull threshold in the released workflow).
- Primitive cells and up-to-20-atom constraints are used for tractability and consistency.
- Lattice preprocessing (e.g., Niggli reduction + matrix handling) improves canonicalization.

### 3.1 Why preprocessing choices matter scientifically

Preprocessing defines the model's effective hypothesis class. For example:

- Restricting to <=20 atoms in the unit cell narrows combinatorial complexity but improves learnability.
- Removing certain elements avoids unsupported chemistry regimes.
- Canonicalization reduces spurious representation variance, improving training signal quality.

### 3.2 Rebuilding the cache (if you train yourself)

Use the data conversion script to serialize CSV data into the training cache format expected by the training datamodule.

In [ ]:
# Example: preprocess MP-20 dataset for training (shell commands)
# !git lfs pull -I data-release/mp-20/ --exclude=""
# !unzip data-release/mp-20/mp_20.zip -d datasets
# !csv-to-dataset --csv-folder datasets/mp_20/ --dataset-name mp_20 --cache-folder datasets/cache

## 4) Training MatterGen: base model and fine-tuning

Training is driven through Hydra configs. Conceptually:

1. Choose data module (`mp_20` or `alex_mp_20`).
2. Train base denoiser (unconditional).
3. Fine-tune by adding property embeddings / adapters.

### 4.1 Base model training

Use base training to learn broad crystal priors before adding target-specific constraints.

In [ ]:
# Base training examples
# !mattergen-train data_module=mp_20 ~trainer.logger
# !mattergen-train data_module=alex_mp_20 ~trainer.logger trainer.accumulate_grad_batches=4

### 4.2 Property fine-tuning

Fine-tuning introduces one or more property embeddings and trains the model to couple structural denoising with property targets.

Methodologically, this is a transfer-learning regime:

- Base model supplies broad chemistry manifold knowledge.
- Fine-tuning steers local sampling trajectories toward target-value regions.

This often improves sample efficiency compared with training conditioned models from scratch.

In [ ]:
# Single-property fine-tuning example
# PROPERTY=dft_mag_density
# !mattergen-finetune adapter.pretrained_name=mattergen_base data_module=mp_20 \
#   +lightning_module/diffusion_module/model/property_embeddings@adapter.adapter.property_embeddings_adapt.$PROPERTY=$PROPERTY \
#   ~trainer.logger data_module.properties=["$PROPERTY"]

## 5) Practical generation workflow

We'll now walk through the actual generation path students use in projects:

1. Pick checkpoint (`mattergen_base` or conditioned model).
2. Generate a batch of candidate structures.
3. Save and inspect outputs (`.extxyz`, zipped `.cif`, optional trajectories).
4. Screen with physics-based or ML-based post-processing.

> For class assignments, start with very small batches (e.g., 8-32 structures) and scale up after validating I/O + analysis scripts.

In [ ]:
from pathlib import Path

results_dir = Path('results/tutorial')
results_dir.mkdir(parents=True, exist_ok=True)
results_dir

In [ ]:
# Unconditional generation example (CLI form)
# !mattergen-generate results/tutorial/unconditional \
#   --pretrained-name=mattergen_base \
#   --batch_size=16 \
#   --num_batches=1

In [ ]:
# Conditional generation example (magnetic density target)
# !mattergen-generate results/tutorial/cond_mag \
#   --pretrained-name=dft_mag_density \
#   --batch_size=16 \
#   --num_batches=1 \
#   --properties_to_condition_on="{'dft_mag_density':0.15}" \
#   --diffusion_guidance_factor=2.0

### 5.1 Interpreting output artifacts

Typical files produced:

- `generated_crystals.extxyz`: multi-frame trajectory of final generated structures (one frame per candidate).
- `generated_crystals_cif.zip`: one CIF per generated crystal.
- `generated_trajectories.zip` (if trajectory recording enabled): full denoising trajectories per sample.

The denoising trajectories are especially useful pedagogically: students can inspect how noisy initial states become ordered periodic structures over sampling steps.

## 6) Loading and visualizing generated materials in Python

Below are lightweight utilities to inspect generated structures directly from `extxyz` and render quick 2D projections.

For publication-quality analysis, you will usually move to tools such as pymatgen symmetry analysis, local environment descriptors, and full relaxation pipelines.

In [ ]:
import ase.io
import matplotlib.pyplot as plt
from ase.visualize.plot import plot_atoms
from pymatgen.io.ase import AseAtomsAdaptor


def load_generated_atoms(extxyz_path):
    # Load all frames from an extxyz file into ASE Atoms objects
    return ase.io.read(extxyz_path, index=':')


def preview_structures_2d(atoms_list, n=3, rotation=('90x,0y,0z')):
    n = min(n, len(atoms_list))
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4))
    if n == 1:
        axes = [axes]
    for i in range(n):
        plot_atoms(atoms_list[i], axes[i], rotation=rotation)
        axes[i].set_title(f'Sample {i}')
        axes[i].set_axis_off()
    plt.tight_layout()
    plt.show()


def ase_to_pymatgen(atoms):
    # Convert ASE Atoms -> pymatgen Structure for advanced analysis
    return AseAtomsAdaptor.get_structure(atoms)

In [ ]:
# Example usage after generation:
# atoms_samples = load_generated_atoms("results/tutorial/unconditional/generated_crystals.extxyz")
# preview_structures_2d(atoms_samples, n=3)
# pmg_structure = ase_to_pymatgen(atoms_samples[0])
# pmg_structure

## 7) Conditioning strategy: experimental design for students

When exploring conditioned generation in coursework, use a controlled protocol:

1. Fix model checkpoint and random seed regime.
2. Sweep target property values (e.g., low/medium/high).
3. Sweep guidance factors (`γ`, e.g., 0.0, 1.0, 2.0, 4.0).
4. Quantify trade-offs: condition adherence vs diversity vs stability proxies.

### 7.1 Practical recommendations

- Start with conservative guidance (`γ≈1-2`).
- Track composition drift under strong guidance.
- Avoid overinterpreting unrelaxed structures; always perform downstream relaxation and validation.
- Compare against unconditional baselines to measure actual conditional benefit.

### 7.2 Example mini-sweep template

In [ ]:
import json
from itertools import product

model_name = "dft_mag_density"
targets = [0.05, 0.15, 0.25]
gammas = [0.0, 1.0, 2.0]

jobs = []
for t, g in product(targets, gammas):
    out = f"results/tutorial/sweep_t{t}_g{g}"
    cmd = (
        f"mattergen-generate {out} --pretrained-name={model_name} "
        f"--batch_size=8 --num_batches=1 "
        f"--properties_to_condition_on=\"{{'dft_mag_density':{t}}}\" "
        f"--diffusion_guidance_factor={g}"
    )
    jobs.append({"target": t, "gamma": g, "command": cmd})

print(json.dumps(jobs, indent=2))
# Execute selected commands manually (or via subprocess) depending on your compute budget.

## 8) Post-generation validation: from ML candidate to chemistry claim

Generative output is **hypothesis generation**, not proof of synthesizability. A robust validation stack typically includes:

1. Structural sanity checks (stoichiometry, interatomic distances, symmetry plausibility).
2. Geometry optimization (MLFF and/or DFT relaxation).
3. Energy-above-hull analysis against a consistent reference scheme.
4. Optional property recalculation with target-fidelity methods (e.g., DFT workflows).

This separation is important for students: MatterGen prioritizes candidate proposal speed, while high-fidelity electronic-structure methods remain the verification backbone.

In [ ]:
# Example metric evaluation call (after obtaining energies):
# !mattergen-evaluate --structures_path=results/tutorial/unconditional \
#   --energies_path=energies.npy \
#   --relax=False \
#   --structure_matcher=disordered \
#   --save_as=metrics

## 9) Suggested exercises for a graduate computational chemistry class

1. **Unconditional baseline study**
   - Generate 256 structures from `mattergen_base`.
   - Compute composition histogram and compare to training dataset priors.

2. **Guidance sensitivity study**
   - Fix a conditioned checkpoint and target value.
   - Evaluate how `γ` changes property adherence and uniqueness.

3. **Chemical system conditioning project**
   - Use `chemical_system` checkpoint with selected element systems.
   - Compare novelty rates across explored vs less-explored systems.

4. **Validation chain assignment**
   - Perform relaxation and stability screening on top-k generated candidates.
   - Report failure modes and uncertainty sources.

These exercises build intuition about where generative AI helps and where physics-based post-processing is indispensable.

## 10) Closing summary

MatterGen combines geometric deep learning and diffusion sampling to generate periodic inorganic crystal candidates at scale. For computational chemistry students, the key mindset is:

- Treat MatterGen as a **proposal engine**.
- Use conditional generation and guidance as **experiment design controls**.
- Use DFT / rigorous evaluation as the **decision-making authority**.

If you continue from this notebook, the next best step is to automate a full pipeline:

`generate -> relax -> evaluate stability/novelty -> property recomputation -> ranked candidate report`.